In [23]:
import json
import pandas as pd
import pyarrow
import requests
from urllib3.exceptions import InsecureRequestWarning
import urllib3

In [24]:
def get_all_datasets():
    # Deshabilitar advertencias por TLS deshabilitado
    urllib3.disable_warnings(InsecureRequestWarning)

    url = "https://sitservicios.lapaz.bo/proxy2.jsp?url=http://gmlpsr00036:8081/dato-abierto/anuario"
    headers = {
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64; rv:149.0) Gecko/20100101 Firefox/149.0",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "es,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br, zstd",
        "Content-Type": "application/json",
        "Referer": "https://sitservicios.lapaz.bo/portal-estadistico/",
        "Origin": "https://sitservicios.lapaz.bo",
        "DNT": "1",
        "Sec-GPC": "1",
        "Sec-Fetch-Dest": "empty",
        "Sec-Fetch-Mode": "cors",
        "Sec-Fetch-Site": "same-origin",
        "Connection": "keep-alive",
    }
    json_data = {}

    resp = requests.post(url, headers=headers, json=json_data, verify=False, timeout=30)
    return resp

In [25]:
def get_dataset_by_id(id=133):
    urllib3.disable_warnings(InsecureRequestWarning)
    url = f"https://sitservicios.lapaz.bo/proxy2.jsp?url=http://gmlpsr00036:8081/dato-abierto/download/{id}&494399419"
    headers_download = {
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64; rv:149.0) Gecko/20100101 Firefox/149.0",
        "Accept": "*/*",
        "Accept-Language": "es,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br, zstd",
        "Referer": "https://sitservicios.lapaz.bo/portal-estadistico/",
        "Content-Type": "text/plain",
        "DNT": "1",
        "Sec-GPC": "1",
        "Sec-Fetch-Dest": "empty",
        "Sec-Fetch-Mode": "cors",
        "Sec-Fetch-Site": "same-origin",
        "Connection": "keep-alive",
    }
    resp = requests.get(url, headers=headers_download, verify=False, timeout=60)
    return resp

In [26]:
response = get_all_datasets()

data = response.json()

df = pd.DataFrame(data)

In [27]:
df.to_parquet("../data/indice.parquet")

In [28]:
lista = df["dab_id"].unique().tolist()

In [29]:
for id in lista:
    dataset_data = get_dataset_by_id(id)
    with open(f"../data/datasets/{id}.csv","w") as file:
        file.write(dataset_data.text)